# Lesson 1. The naive baseline: a pretrained encoder plus a head

The intensity feature failed because it discarded the image. The naive fix is
common: take a network **pretrained on ImageNet** (millions of natural photos, such
as cats and cars), freeze it, and use its features. The network has never seen an
MRI. That mismatch is why this is the naive baseline, not the best result.

You use **`resnet18`**. Feed it a few central sagittal slices (grayscale copied to 3
channels, ImageNet-normalized). Mean-pool its 512-d features per study. Then fit the
same head from Lesson 0. Targets: **effusion** and **acl**.

**Where you are: Lesson 1 of 3.** You have the tools from Lesson 0. Now you give
them a real feature, a frozen ImageNet encoder. Check two results: whether the score
is above the shuffle control, and how far it is below the tuned pipeline in this
repo.

Go deeper on the encoder: [torchvision models](https://pytorch.org/vision/stable/models.html),
[transfer learning](https://cs231n.github.io/transfer-learning/).

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# config: read a bounded sample, never the full ~570 GB ----------------------
N_STUDIES = 300     # read only this many studies' DICOMs, not the full dataset
K_SLICES  = 5       # central slices per study
SEED      = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# This lesson needs the RSNA competition data, which lives on Kaggle. The exact
# mount folder can vary, so we try the known layouts with shallow checks (no walk
# of the full 570 GB tree). Set RSNA_DATA_ROOT to run with a local copy.
CANDIDATE_ROOTS = [r for r in [
    os.environ.get("RSNA_DATA_ROOT"),
    "/kaggle/input/rsna-knee-abnormality-detection",
    "/kaggle/input/competitions/rsna-knee-abnormality-detection",
] if r]

def _first_file(paths):
    for p in paths:
        if os.path.isfile(p):
            return p
    return None

def _first_dir(paths):
    for p in paths:
        if os.path.isdir(p):
            return p
    return None

def _resolve_rsna():
    for root in CANDIDATE_ROOTS:
        train = _first_file([f"{root}/train.csv", f"{root}/metadata/train.csv"])
        series = _first_file([f"{root}/train_series.csv", f"{root}/metadata/train_series.csv"])
        sdir = _first_dir([f"{root}/train_series", f"{root}/sample/train_series"])
        if train and series and sdir:
            return train, series, sdir
    return None

_resolved = _resolve_rsna()
if _resolved is None:
    raise SystemExit(
        "This lesson needs the RSNA Knee MRI competition data, which is on Kaggle. "
        "Open this notebook in Kaggle, join the competition at "
        "kaggle.com/competitions/rsna-knee-abnormality-detection, then use Add Input "
        "to attach it. It cannot run on Colab, which has no /kaggle/input."
    )
TRAIN_CSV, SERIES_CSV, TRAIN_SERIES_DIR = _resolved
print("labels :", TRAIN_CSV)
print("series :", SERIES_CSV)
print("dicoms :", TRAIN_SERIES_DIR)

In [ ]:
import pydicom

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def list_studies(train_series_dir, n=None):
    studies = sorted(d for d in os.listdir(train_series_dir)
                     if os.path.isdir(os.path.join(train_series_dir, d)))
    return studies[:n] if n else studies

def first_sagittal_series(series_df, study, train_series_dir):
    study_dir = os.path.join(train_series_dir, study)
    rows = series_df[(series_df["StudyInstanceUID"] == study) &
                     (series_df["Anatomical_Plane"].astype(str).str.lower() == "sagittal")]
    for sid in rows["SeriesInstanceUID"].astype(str):
        sdir = os.path.join(study_dir, sid)
        if os.path.isdir(sdir) and glob.glob(os.path.join(sdir, "*.dcm")):
            return sdir
    for sdir in sorted(glob.glob(os.path.join(study_dir, "*"))):
        if glob.glob(os.path.join(sdir, "*.dcm")):
            return sdir
    return None

def _decode(ds):
    px = np.asarray(ds.pixel_array, dtype=np.float32)
    px = px * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")) == "MONOCHROME1":
        px = float(px.max() + px.min()) - px
    return px

def load_central_slices(series_dir, k=K_SLICES):
    recs = []
    for i, p in enumerate(sorted(glob.glob(os.path.join(series_dir, "*.dcm")))):
        try:
            ds = pydicom.dcmread(p)
        except Exception:
            continue
        try:
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            coord = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            coord = float(i)
        try:
            px = _decode(ds)
        except Exception:
            continue
        if px.ndim == 2:
            recs.append((coord, px))
    if not recs:
        return []
    recs.sort(key=lambda r: r[0])
    lo = max(0, len(recs) // 2 - k // 2)
    return [px for _, px in recs[lo:lo + k]]

def _norm01(img):
    finite = img[np.isfinite(img)]
    if finite.size == 0:
        return np.zeros_like(img, dtype=np.float32)
    lo, hi = np.percentile(finite, [1, 99])
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    clean = np.nan_to_num(img, nan=float(lo), posinf=float(hi), neginf=float(lo))
    return np.clip((clean - lo) / (hi - lo), 0, 1).astype(np.float32)

def prep_batch(slices, size=224):
    frames = []
    for s in slices:
        t = torch.from_numpy(_norm01(s))[None, None]
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        frames.append(t[0, 0])
    x = torch.stack(frames)[:, None].repeat(1, 3, 1, 1)
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x - mean) / std

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

def cv_auc(X, y, seed=SEED, n_splits=5, shuffle_labels=False):
    """Out-of-fold ROC-AUC. Each study is one row, so folds never split a study."""
    y = np.asarray(y).astype(int)
    if shuffle_labels:
        y = y[np.random.default_rng(seed).permutation(len(y))]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr, va in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=1000).fit(sc.transform(X[tr]), y[tr])
        oof[va] = clf.predict_proba(sc.transform(X[va]))[:, 1]
    return float(roc_auc_score(y, oof))

def mean_auc(X, y, shuffle_labels=False, seeds=range(10)):
    """Average AUC over several CV seeds -- stable when N is small and noisy."""
    return float(np.mean([cv_auc(X, y, seed=s, shuffle_labels=shuffle_labels)
                          for s in seeds]))

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

def build_resnet18():
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = torch.nn.Identity()          # 512-d penultimate features
    return m.eval().to(device)

@torch.inference_mode()
def resnet18_features(model, batch):
    return model(batch.to(device)).float().cpu()   # [k, 512]

def study_feature(model, feats_fn, slices):
    return feats_fn(model, prep_batch(slices)).mean(0).numpy()  # mean-pool slices

## Step 1: Build features and score

Read up to `N_STUDIES`, extract ResNet-18 features, and cross-validate effusion and
acl against their shuffle controls.

In [ ]:
labels = pd.read_csv(TRAIN_CSV); series = pd.read_csv(SERIES_CSV)
lab = labels.set_index("StudyInstanceUID")

model = build_resnet18()
studies = list_studies(TRAIN_SERIES_DIR, n=N_STUDIES)
X, kept = [], []
for s in studies:
    sd = first_sagittal_series(series, s, TRAIN_SERIES_DIR)
    if sd is None:
        continue
    sl = load_central_slices(sd)
    if not sl:
        continue
    X.append(study_feature(model, resnet18_features, sl)); kept.append(s)
X = np.stack(X)
print("feature matrix:", X.shape, "on", len(kept), "studies")

for tgt in ["Effusion", "ACL"]:
    y = lab.loc[kept, tgt].to_numpy()
    real = cv_auc(X, y); shuf = cv_auc(X, y, shuffle_labels=True)
    print(f"{tgt:9s} real AUC={real:.3f}  shuffle AUC={shuf:.3f}  "
          f"(+{int(y.sum())}/-{int(len(y)-y.sum())})")

## Step 2: Read the result honestly

Two results to notice:

- **Above the shuffle control.** Real signal exists. The frozen ImageNet features
  are from the wrong domain, but they still carry some information about the knee.
- **Far below the tuned pipeline.** The tuned pipeline in this repo reaches AUC near
  0.92 (effusion) and 0.87 (acl). Your naive score is much lower. To close that gap
  you need domain-matched encoders, better slice selection, and real tuning. That is
  the rest of the course.

Your score also **varies** when you change `N_STUDIES` or `SEED`. On a few hundred
studies that variation is large. The next experiment shows it, and Lesson 2 makes it
the main topic.

## Experiment: how much does the naive score vary?

Same features, same studies. Change only the seed and read the effusion AUC six
times. The spread is the error bar you did not print. Any single run, including one
you would post to a leaderboard, is one sample from this range.

In [ ]:
y_eff = lab.loc[kept, "Effusion"].to_numpy()
scores = [cv_auc(X, y_eff, seed=s) for s in range(6)]
print("effusion AUC across seeds:", [round(s, 3) for s in scores])
print(f"spread: {min(scores):.3f} to {max(scores):.3f}  (range {max(scores)-min(scores):.3f})")

## Step 3: A note on leakage

`train.csv` includes a free-text **`Report`**, the radiologist write-up. To use it
here is a common and serious mistake. Fit a bag-of-words model on the report text
alone and the AUC increases toward 1.0. The cause is not a good vision model. The
report contains the word *effusion*. That is **leakage**: a feature that encodes the
answer.

The report is not available at test time in the way you use it here. Even when text
is allowed, a feature that contains the label description gives a score you cannot
reproduce in the real task. A high score is a question, not proof of skill. Always
ask why it is high.

Go deeper: [common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline

y = lab.loc[kept, "Effusion"].to_numpy()
reports = lab.loc[kept, "Report"].fillna("").astype(str).to_numpy()

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(y))
for tr, va in skf.split(reports, y):
    clf = make_pipeline(TfidfVectorizer(min_df=2), LogisticRegression(max_iter=1000))
    clf.fit(reports[tr], y[tr])
    oof[va] = clf.predict_proba(reports[va])[:, 1]
print("Report-text AUC (effusion):", round(roc_auc_score(y, oof), 3),
      "<- very high AUC = leakage, not skill")

## Recap

A frozen ImageNet encoder scores above the shuffle control but stays below the
tuned pipeline. It is an honest, reproducible starting point. The near-perfect
report result was a warning, not a real score. Next, in Lesson 2, you hold the data
fixed and swap the encoder. Does a newer or larger one help?